In [1]:
using VeryDiff, JLD2, Polynomials
using VeryDiff: Dense, ChebyshevPoly, Network

Set parameter Username
Set parameter LicenseID to value 2695178
Academic license - for non-commercial use only - expires 2026-08-12


In [2]:
params = load("simple_nn.jld2")["params"]
#params= load("just_relu.jld2")["params"]

5-element Vector{Any}:
 ("linear", [0.5 0.5; 0.5 -0.5], [0.0, 0.0])
 ("chebyshev", [0.6220084679281463 1.0000000000000002 … -0.1220084679281462 -5.551115123125783e-18; 0.6220084679281463 1.0000000000000002 … -0.1220084679281462 -5.551115123125783e-18], [-1.0, -1.0], [1.0, 1.0])
 ("linear", [0.5230731272178457 0.5230731272178457; 0.5230731272178457 -0.5230731272178457], [-0.5236458801223454, 0.0])
 ("chebyshev", [0.8717732763416244 1.25972925015921 … -0.06853362124847143 0.06792266583018995; 0.5945712325507181 0.9558892896284533 … -0.11662658773649151 -5.828670879282072e-17], [-1.0, -1.0], [1.0, 1.0])
 ("linear", [-1.0 1.0; 0.0 1.0], [3.0, 0.0])

In [3]:
W1, b1 = params[1][2], params[1][3]
W2, b2 = params[3][2], params[3][3]
W3, b3 = params[5][2], params[5][3]

coeffs1, l1, u1 = params[2][2], params[2][3], params[2][4]
coeffs2, l2, u2 = params[4][2], params[4][3], params[4][4]

nn_poly = Network([
    Dense(W1, b1), 
    ChebyshevPoly(coeffs1, l1, u1),
    Dense(W2, b2),
    ChebyshevPoly(coeffs2, l2, u2),
    Dense(W3, b3)
])

Network(Layer[Dense{Float64, Matrix{Float64}, Vector{Float64}}([0.5 0.5; 0.5 -0.5], [0.0, 0.0]), ChebyshevPoly{Float64, Vector{Float64}}([0.6220084679281463 1.0000000000000002 … -0.1220084679281462 -5.551115123125783e-18; 0.6220084679281463 1.0000000000000002 … -0.1220084679281462 -5.551115123125783e-18], [-1.0, -1.0], [1.0, 1.0]), Dense{Float64, Matrix{Float64}, Vector{Float64}}([0.5230731272178457 0.5230731272178457; 0.5230731272178457 -0.5230731272178457], [-0.5236458801223454, 0.0]), ChebyshevPoly{Float64, Vector{Float64}}([0.8717732763416244 1.25972925015921 … -0.06853362124847143 0.06792266583018995; 0.5945712325507181 0.9558892896284533 … -0.11662658773649151 -5.828670879282072e-17], [-1.0, -1.0], [1.0, 1.0]), Dense{Float64, Matrix{Float64}, Vector{Float64}}([-1.0 1.0; 0.0 1.0], [3.0, 0.0])])

In [4]:
nn_poly.layers[2].coeffs

2×6 Matrix{Float64}:
 0.622008  1.0  0.455342  1.249e-17  -0.122008  -5.55112e-18
 0.622008  1.0  0.455342  1.249e-17  -0.122008  -5.55112e-18

In [5]:
nn_poly.layers[2]([0.3, -0.1])

2-element Vector{Float64}:
  0.5065596711521083
 -0.03657189457634058

In [6]:
p = ChebyshevT(nn_poly.layers[2].coeffs[1,:])

@show p(0.3)
@show p(-0.1);

p(0.3) = 0.5065596711521084
p(-0.1) = -0.03657189457634058


# Execution just for Linear Layer

In [8]:
nn_poly.layers[5].W * [-0.0295876, 0.1891] + nn_poly.layers[5].b

2-element Vector{Float64}:
 3.2186876
 0.1891

In [9]:
nn_poly.layers[5]([-0.0295876, 0.1891])

2-element Vector{Float64}:
 3.2186876
 0.1891

In [10]:
nn_poly.layers[5]([0.1, 0.2])

2-element Vector{Float64}:
 3.1
 0.2

In [11]:
nn_poly([0.1, 0.2])

2-element Vector{Float64}:
 3.2275999340023755
 0.18886515635793877

# Step-by-Step Execution for Example Input

In [13]:
x1 = nn_poly.layers[1]([0.1, 0.2])
x2 = nn_poly.layers[2](x1)
x3 = nn_poly.layers[3](x2)
x4 = nn_poly.layers[4](x3)
x5 = nn_poly.layers[5](x4)

@show x1
@show x2
@show x3
@show x4
@show x5;

x1 = [0.15000000000000002, -0.05]
x2 = [0.23661596972724458, -0.0006310233200055304]
x3 = [-0.4002084962287751, 0.12409752657625561]
x4 = [-0.03873477764443667, 0.18886515635793877]
x5 = [3.2275999340023755, 0.18886515635793877]


In [16]:
p2 = ChebyshevT(nn_poly.layers[2].coeffs[1,:])
# we don't need a second polynomial here. Coefficients for both neurons are the same 
# but this is not guaranteed for layers in general!

@show p2(x1[1])
@show p2(x1[2]);

p2(x1[1]) = 0.23661596972724458
p2(x1[2]) = -0.0006310233200054749


In [17]:
p41 = ChebyshevT(nn_poly.layers[4].coeffs[1,:])
p42 = ChebyshevT(nn_poly.layers[4].coeffs[2,:])

@show p41(x3[1])
@show p42(x3[2]);

p41(x3[1]) = -0.03873477764443667
p42(x3[2]) = 0.18886515635793882
